# Transformers con Hugging Face

Vamos a usar la librería **Hugging Face Transformers** para probar modelos
preentrenados que hacen cosas sorprendentes con **3 líneas de código**.

### Ejemplos que vamos a hacer
1. Análisis de sentimiento
2. Clasificación sin entrenar (zero-shot)
3. Preguntas y respuestas
4. Resumen de textos
5. Traducción
6. Generación de texto
7. Reconocimiento de entidades (NER)

Todo sin entrenar nada: son modelos que ya vienen preparados.

> ⚠️ En Colab, activa la GPU: *Entorno de ejecución → Cambiar tipo → T4 GPU*.
> Algunos modelos cargan más rápido con GPU.

## Instalación

In [7]:
#!pip install transformers -q

In [1]:
from transformers import pipeline
print('Hugging Face Transformers listo')
print('La función pipeline() carga modelos preentrenados automáticamente')

Hugging Face Transformers listo
La función pipeline() carga modelos preentrenados automáticamente


---
## 1. Análisis de sentimiento

El modelo lee un texto y decide si es **positivo** o **negativo**.
Internamente usa un Transformer tipo BERT entrenado con miles de reseñas.

In [9]:
sentimiento = pipeline("text-classification", model="tabularisai/robust-sentiment-analysis")

textos = [
    'I love this product, it is amazing!',
    'This was terrible, worst experience ever.',
    'The weather is nice today.',
    'I am not sure if I liked it or not.',
]

for texto in textos:
    resultado = sentimiento(texto)[0]
    print(f'  "{texto}"')
    print(f'  → {resultado["label"]}  ({resultado["score"]:.0%})\n')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  "I love this product, it is amazing!"
  → Very Positive  (43%)

  "This was terrible, worst experience ever."
  → Very Negative  (86%)

  "The weather is nice today."
  → Neutral  (57%)

  "I am not sure if I liked it or not."
  → Neutral  (59%)



---
## 2. Sentimiento en español (modelo multilingüe)

Podemos usar un modelo entrenado específicamente en varios idiomas.

In [10]:
sentimiento_es = pipeline('sentiment-analysis',
                          model='nlptown/bert-base-multilingual-uncased-sentiment')

textos_es = [
    'Este producto es excelente, me encanta!',
    'Horrible, no funciona nada',
    'Está bien, nada especial',
    'Lo mejor que he comprado en años',
]

for texto in textos_es:
    resultado = sentimiento_es(texto)[0]
    estrellas = resultado['label']
    print(f'  "{texto}"')
    print(f'  → {estrellas}  ({resultado["score"]:.0%})\n')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  "Este producto es excelente, me encanta!"
  → 5 stars  (88%)

  "Horrible, no funciona nada"
  → 1 star  (98%)

  "Está bien, nada especial"
  → 3 stars  (60%)

  "Lo mejor que he comprado en años"
  → 5 stars  (87%)



---
## 3. Clasificación sin entrenar (Zero-Shot)

Esto es impresionante: el modelo clasifica textos en categorías
que **tú le defines en ese momento**, sin haber sido entrenado para ellas.
Es como pedirle que entienda cualquier categoría al vuelo.

In [11]:
clasificador = pipeline('zero-shot-classification',
                        model='facebook/bart-large-mnli')

texto = 'Ayer fui al dentista porque me dolía una muela'
categorias = ['salud', 'deportes', 'tecnología', 'cocina', 'viajes']

resultado = clasificador(texto, candidate_labels=categorias)

print(f'Texto: "{texto}"\n')
for label, score in zip(resultado['labels'], resultado['scores']):
    barra = '█' * int(score * 30)
    print(f'  {label:12s}  {score:.0%}  {barra}')

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Texto: "Ayer fui al dentista porque me dolía una muela"

  salud         92%  ███████████████████████████
  viajes        4%  █
  deportes      2%  
  tecnología    1%  
  cocina        1%  


In [12]:
# Prueba con otro texto y otras categorías
texto2 = 'El nuevo iPhone tiene una cámara increíble'
cats2 = ['tecnología', 'moda', 'alimentación', 'política']

r2 = clasificador(texto2, candidate_labels=cats2)
print(f'Texto: "{texto2}"\n')
for label, score in zip(r2['labels'], r2['scores']):
    barra = '█' * int(score * 30)
    print(f'  {label:15s}  {score:.0%}  {barra}')

Texto: "El nuevo iPhone tiene una cámara increíble"

  tecnología       82%  ████████████████████████
  moda             13%  ███
  política         4%  █
  alimentación     2%  


---
## 4. Preguntas y respuestas

Le das un **contexto** (un párrafo) y una **pregunta**, y el modelo
encuentra la respuesta dentro del texto. No inventa: extrae.

In [17]:

qa = pipeline("question-answering", model="mrm8488/distilroberta-finetuned-squadv1")

contexto = """Madrid es la capital de España. Está ubicada en el centro 
de la península ibérica y tiene más de 3 millones de habitantes. 
El estadio Santiago Bernabéu es la casa del Real Madrid, uno de 
los clubes de fútbol más famosos del mundo."""

preguntas = [
    '¿Cuál es la capital de España?',
    '¿Cuántos habitantes tiene Madrid?',
    '¿Cómo se llama el estadio?',
    '¿De qué equipo es el Bernabéu?'
]

print('Contexto:', contexto[:80], '...\n')
for pregunta in preguntas:
    r = qa(question=pregunta, context=contexto)
    print(f'  P: {pregunta}')
    print(f'  R: {r["answer"]}  (confianza: {r["score"]:.0%})\n')

config.json:   0%|          | 0.00/524 [00:00<?, ?B/s]

c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Diego Nuñez\.cache\huggingface\hub\models--mrm8488--distilroberta-finetuned-squadv1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: mrm8488/distilroberta-finetuned-squadv1
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Contexto: Madrid es la capital de España. Está ubicada en el centro 
de la península ibéri ...

  P: ¿Cuál es la capital de España?
  R: Madrid  (confianza: 99%)

  P: ¿Cuántos habitantes tiene Madrid?
  R: tiene más de 3 millones de habitantes.  (confianza: 12%)

  P: ¿Cómo se llama el estadio?
  R: El estadio Santiago Bernabéu  (confianza: 1%)

  P: ¿De qué equipo es el Bernabéu?
  R: Real Madrid,  (confianza: 11%)



---
## 5. Resumen de textos

Le pasas un texto largo y te devuelve un **resumen** más corto.
Internamente genera texto nuevo que condensa la información.

In [5]:
resumidor = pipeline("text-generation", model="sshleifer/distilbart-cnn-12-6")

texto_largo = """Artificial intelligence has made significant strides in recent years, 
particularly in the field of natural language processing. The introduction of the 
Transformer architecture in 2017 marked a turning point, enabling models to process 
text in parallel rather than sequentially. This led to the development of powerful 
models like BERT, which excels at understanding context, and GPT, which can generate 
remarkably human-like text. The release of ChatGPT in late 2022 brought these 
capabilities to the mainstream, with millions of users interacting with AI for the 
first time. Today, large language models are being used in education, healthcare, 
software development, and countless other fields."""

resumen = resumidor(texto_largo, max_length=60, min_length=20)
print('TEXTO ORIGINAL:')
print(texto_largo[:200], '...\n')
print('RESUMEN:')
print(resumen[0]['summary_text'])

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Diego Nuñez\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Please make sure the generation config includes `forced_bos_token_id=0`.

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

BartForCausalLM LOAD REPORT from: sshleifer/distilbart-cnn-12-6
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
final_logits_bias                                         | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.embed_positions.weight                      | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encode

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'min_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


TEXTO ORIGINAL:
Artificial intelligence has made significant strides in recent years, 
particularly in the field of natural language processing. The introduction of the 
Transformer architecture in 2017 marked a turn ...

RESUMEN:


KeyError: 'summary_text'

---
## 6. Generación de texto

Le das un inicio de frase y el modelo la **continúa** generando texto
coherente. Esto es lo que hace GPT internamente.

In [4]:
generador = pipeline('text-generation', model='gpt2')

inicios = [
    'Artificial intelligence will',
    'The best way to learn deep learning is',
]

for inicio in inicios:
    resultado = generador(inicio, max_length=50, num_return_sequences=1,
                          truncation=True)
    print(f'  Inicio: "{inicio}"')
    print(f'  Generado: {resultado[0]["generated_text"]}\n')
    print()

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Diego Nuñez\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Inicio: "Artificial intelligence will"
  Generado: Artificial intelligence will eventually become a huge problem in the real world. The question now is whether it will be possible to take advantage of the fact the "human brain" remains intact for decades to come.

According to a new paper published in the Proceedings of the National Academy of Sciences, "The ability to control its activity is a major hurdle to achieving a truly advanced intelligence."

Researchers took a test of mice with a control group of about three years old. They then took a larger group of mice and put them in a group of 40.

The mice were given a series of artificial intelligence tasks (the mice were randomly assigned to one of the three groups, the controls were given the other two, and the researchers watched their brains respond to the artificial intelligence task).

"The fact that we could use the human brain to control its activity is a real challenge because it does not exist in the human brain," said co

---
## 8. Reconocimiento de entidades (NER)

El modelo identifica **personas**, **lugares**, **organizaciones** y otras
entidades dentro del texto.

In [3]:
ner = pipeline('ner', aggregation_strategy="simple")

texto_ner = 'Elon Musk founded SpaceX in California and also runs Tesla.'

entidades = ner(texto_ner)
print(f'Texto: "{texto_ner}"\n')
print('Entidades encontradas:')
for e in entidades:
    print(f'  {e["word"]:15s}  tipo: {e["entity_group"]:5s}  confianza: {e["score"]:.0%}')

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texto: "Elon Musk founded SpaceX in California and also runs Tesla."

Entidades encontradas:
  Elon Musk        tipo: PER    confianza: 100%
  SpaceX           tipo: ORG    confianza: 100%
  California       tipo: LOC    confianza: 100%
  Tesla            tipo: ORG    confianza: 99%


---
## 9. Tabla resumen: pipeline de Hugging Face

| Tarea | `pipeline(...)` | Qué hace |
|-------|----------------|----------|
| Sentimiento | `'sentiment-analysis'` | ¿Positivo o negativo? |
| Zero-shot | `'zero-shot-classification'` | Clasifica en categorías al vuelo |
| Q&A | `'question-answering'` | Responde preguntas desde un contexto |
| Resumen | `'summarization'` | Texto largo → corto |
| Traducción | `'translation'` | De un idioma a otro |
| Generación | `'text-generation'` | Continúa un texto |
| NER | `'ner'` | Detecta personas, lugares, organizaciones |

### La idea clave
```python
# Esto es todo lo que necesitas:
from transformers import pipeline
mi_modelo = pipeline('tarea')         # carga el modelo
resultado = mi_modelo('mi texto')     # úsalo
```

3 líneas. Miles de modelos disponibles en [huggingface.co](https://huggingface.co).